### **Machine Learning em Escala e Sistemas de Recomendação (MLlib)**
**O Cenário Real**: Plataformas de streaming ou bancos precisam prever comportamentos em tempo real (ex: calcular o risco de crédito de um cliente ou recomendar o próximo filme).

**Como o Spark atua**: Através do MLlib, o Spark treina **modelos estatísticos distribuídos**. Como ele mantém as matrizes de cálculo e iterações em cache na **memória RAM**, ele evita o gargalo de gravar e ler dados do disco a cada ciclo de aprendizado.

**Por que é relevante**: Em Machine Learning tradicional, treinar milhões de registros **trava a máquina**. No Spark, o treinamento é distribuído entre vários nós do cluster de forma **paralela**.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

def main():
    spark = (SparkSession.builder
             .appName("Modulo_2_MLlib")
             .master("local[*]")
             .getOrCreate())
    
    spark.sparkContext.setLogLevel("WARN")
    print("=========================================================")
    print("=== PILAR 2: SPARK MLLIB (MACHINE LEARNING DISTRIBUIDO) ===")
    print("=========================================================")

    sales_data = [
        ("GOOG", "Sam", 200.0),
        ("GOOG", "Charlie", 120.0),
        ("GOOG", "Frank", 340.0),
        ("MSFT", "Tina", 600.0),
        ("MSFT", "Amy", 124.0),
        ("MSFT", "Vanessa", 243.0),
        ("FB", "Carl", 870.0),
        ("FB", "Sarah", 350.0),
        ("APPL", "John", 250.0),
        ("APPL", "Linda", 130.0),
        ("APPL", "Mike", 750.0),
        ("APPL", "Chris", 350.0)
    ]
    df = spark.createDataFrame(sales_data, ["Company", "Person", "Sales"])

    # Preparando o Vetor de Caracteristicas (Features)
    assembler = VectorAssembler(inputCols=["Sales"], outputCol="features")
    df_ml = assembler.transform(df)

    # Definindo a Variavel Alvo (Target ficticio: Meta = Sales * 1.1)
    df_ml = df_ml.withColumn("target", col("Sales") * 1.1)

    # Treinando a Regressao Linear em Memoria RAM
    lr = LinearRegression(featuresCol="features", labelCol="target")
    model = lr.fit(df_ml)

    print(f"\nModelo Treinado!")
    print(f"Coeficiente: {model.coefficients[0]:.2f} | Intercepto: {model.intercept:.2f}")

    print("\n--- Predicoes Geradas pelo Modelo ---")
    predictions = model.transform(df_ml)
    predictions.select("Person", "Sales", "prediction").show(5)

    spark.stop()

if __name__ == "__main__":
    main()

=== PILAR 2: SPARK MLLIB (MACHINE LEARNING DISTRIBUIDO) ===

Modelo Treinado!
Coeficiente: 1.10 | Intercepto: -0.00

--- Predicoes Geradas pelo Modelo ---
+-------+-----+------------------+
| Person|Sales|        prediction|
+-------+-----+------------------+
|    Sam|200.0|219.99999999999977|
|Charlie|120.0|131.99999999999972|
|  Frank|340.0| 373.9999999999999|
|   Tina|600.0| 660.0000000000001|
|    Amy|124.0|136.39999999999972|
+-------+-----+------------------+
only showing top 5 rows
